In [3]:
import pandas as pd
import cellbell
import requests
import json
import threading
import os
import time
from requests.utils import quote
pd.options.mode.chained_assignment = None

In [4]:


# ## TempDF is the repository of all Query Results
# Every query collects the DOI, PII (for science Direct), the title and the Journal Name


global TempDF

TempDF=pd.DataFrame(columns=['Query','PII','DOI','Title', 'Journal'])
TempDF=TempDF.set_index('DOI')

In [7]:
# list of issn numbers of journals
jlist = ['0950-0618']


In [ ]:
for journal_issn in jlist:
    # for dates  in [('2015-01','2020-01')]:
    for dates  in [('1999-01','2005-01'),('2005-01','2010-01'),('2010-01','2015-01'),('2015-01','2020-01'),('2020-01','2024-01')]:
        global TempDF

        TempDF=pd.DataFrame(columns=['Query','PII','DOI','Title', 'Journal'])
        TempDF=TempDF.set_index('DOI')
        
        start_date = dates[0]
        end_date =  dates[1]
        def cross_reference_search(query,TempDF):

            cursor = "*"
            keep_paging = True
            max_rows = 1000


            base_url = 'https://api.crossref.org/works?query='

            headers = {
                   'Accept': 'application/json',
                   'mailto':   'cez198233@iitd.ac.in'
                 }

            # params = {
            # 'filter': f'from-pub-date:{start_date},container-title:{journal_name}',
            #     }
            params = {
            'filter': f'from-pub-date:{start_date},until-pub-date:{end_date},issn:{journal_issn}',
                }
            while (keep_paging):

                try:
                    # https://api.crossref.org/works?query=+&filter=from-pub-date%3A2019-01%2Ccontainer-title%3ACeramics+International&rows=1000&cursor=%2A

                    r = requests.get(base_url + query + "&rows=" + str(max_rows) + "&cursor=" + cursor,
                                              headers=headers, timeout=100,params=params)

                    cursor = quote(r.json()['message']['next-cursor'], safe='')

                    if len(r.json()['message']['items']) == 0:
                                    keep_paging = False

                    for item in r.json()['message']['items']:
                        try:
                            Journal=item['container-title'][0]
                        except:
                            print('yahan')
                            Journal='None'

                        TempDF.loc[item['DOI']]=(query,'None',item['title'][0],Journal)
                        if TempDF.shape[0]%500 == 0:
                            time.sleep(1)
                            # pass
                            # print('Papers found = %d'%(int(TempDF.shape[0])))

                        # print('ek mila')


                except Exception as e:
                    print(e)
                    keep_paging = False

            return None
        cross_reference_search(' ',TempDF)
        s1 = start_date.split('-')[0]
        s2 = end_date.split('-')[0]
        TempDF.to_csv(f'{journal_issn}_{s1}_{s2}.csv')
        print('done',journal_issn,len(TempDF),s1,s2)
        time.sleep(5)
    cellbell.ding()

done 0950-0618 389 1999 2005
done 0950-0618 1300 2005 2010
done 0950-0618 4391 2010 2015
done 0950-0618 11403 2015 2020
